In [ ]:
from dataclasses import dataclass
from typing import Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
def make_windows(
    series: np.ndarray,
    lookback: int,
    horizon: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    series: shape (T,) for univariate or (T, C) for multivariate
    Returns:
      X: (N, lookback, C)
      y: (N, horizon, C)  (you can slice to a target channel if you want)
    """
    series = np.asarray(series, dtype=np.float32)
    if series.ndim == 1:
        series = series[:, None]  # (T, 1)

    T, C = series.shape
    N = T - lookback - horizon + 1
    if N <= 0:
        raise ValueError("Time series too short for given lookback/horizon.")

    X = np.zeros((N, lookback, C), dtype=np.float32)
    y = np.zeros((N, horizon, C), dtype=np.float32)

    for i in range(N):
        X[i] = series[i : i + lookback]
        y[i] = series[i + lookback : i + lookback + horizon]

    return X, y


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X)  # (N, L, C)
        self.y = torch.from_numpy(y)  # (N, H, C)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
class MLPForecaster(nn.Module):
    """
    MLP for direct forecasting:
      input:  (B, lookback, C) -> flatten -> MLP -> (B, horizon, out_dim)
    """
    def __init__(
        self,
        lookback: int,
        in_channels: int,
        horizon: int,
        out_dim: int = 1,                 # often 1 for a single target variable
        hidden_sizes=(256, 256),
        dropout: float = 0.1,
        activation: str = "relu",
    ):
        super().__init__()
        self.lookback = lookback
        self.in_channels = in_channels
        self.horizon = horizon
        self.out_dim = out_dim

        act = {"relu": nn.ReLU, "gelu": nn.GELU, "tanh": nn.Tanh}[activation]

        layers = []
        prev = lookback * in_channels
        for hs in hidden_sizes:
            layers += [nn.Linear(prev, hs), act(), nn.Dropout(dropout)]
            prev = hs
        layers += [nn.Linear(prev, horizon * out_dim)]

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, L, C)
        b = x.size(0)
        x = x.view(b, -1)  # flatten
        y = self.net(x)    # (B, H*out_dim)
        return y.view(b, self.horizon, self.out_dim)

In [ ]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# FEATURE ENGINEERING (EXOGENOUS VARIABLES)
# -----------------------------------------------------------------------------
# 1. Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df[DATE_COL].dt.dayofweek

# 2. Month (1=January, 12=December)
df['month'] = df[DATE_COL].dt.month

# 3. Day of month (1-31) - captures paydays
df['day_of_month'] = df[DATE_COL].dt.day

# 4. Is Weekend (Binary)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

# Christmas Day flag (1 on Dec 25, else 0)
df['is_christmas_day'] = ((df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)).astype(float)

# 5. Promotions
# Identify all columns that start with 'promo_'
promo_cols = [col for col in df.columns if col.startswith('promo_')]
print(f"Found {len(promo_cols)} promotion columns: {promo_cols}")

# Ensure promo columns are numeric (float)
for col in promo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)

# Define the list of exogenous features to use
#EXOG_COLS = None
EXOG_COLS = ['day_of_week', 'month', 'day_of_month', 'is_weekend', 'is_christmas_day'] + promo_cols




In [ ]:
@dataclass
class TrainConfig:
    lookback: int = 48
    horizon: int = 153
    batch_size: int = 32
    train_size: int = 453
    val_size : int = 153
    lr: float = 1e-3
    epochs: int = 30
    weight_decay: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# mlp_timeseries_forecasting.py
# A minimal MLP forecaster for time series using sliding windows (PyTorch)

from dataclasses import dataclass
from typing import Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader




def train_mlp_forecaster(
    series: np.ndarray,
    cfg: TrainConfig,
    target_channel: int = 0,    # which channel to forecast
    val_ratio: float = 0.2,
    hidden_sizes=(64, 32),
    target: str = TARGET_COL,
):
    """
    series: (T,) or (T, C)
    Trains an MLP on scaled data. Returns (model, scaler).
    """
    total_train_val = cfg.train_size + cfg.val_size
    
    total_train_val = cfg.train_size + cfg.val_size
    train_slice = slice(-(total_train_val+cfg.horizon), -(cfg.val_size+cfg.horizon))
    val_slice = slice(-(cfg.val_size+cfg.horizon), -cfg.horizon)
    test_slice = slice(-cfg.horizon, None)
    
    train = df[target][train_slice].values
    val = df[target][val_slice].values
    test = df[target][test_slice].values
    
    #scaler = MinMaxScaler()
    scaler = RobustScaler()
    
    # Fit scaler on train data
    train_scaled = scaler.fit_transform(train.reshape(-1, 1))
    
    # Validation needs preceding `lookback` data from train to form the first window correctly
    val_context = np.concatenate([train[-cfg.lookback:], val])
    val_scaled = scaler.transform(val_context.reshape(-1, 1))
    
    # Create windows separately to completely avoid data leakage
    X_train, y_train_full = make_windows(train_scaled, cfg.lookback, cfg.horizon)
    X_val, y_val_full = make_windows(val_scaled, cfg.lookback, cfg.horizon)
    
    # y_target: (N, H, 1)
    y_train = y_train_full[:, :, target_channel : target_channel + 1]
    y_val = y_val_full[:, :, target_channel : target_channel + 1]

    # Dynamically find number of channels for model definition
    C = X_train.shape[2]

    train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(WindowDataset(X_val, y_val), batch_size=cfg.batch_size, shuffle=False)
        
    model = MLPForecaster(
        lookback=cfg.lookback,
        in_channels=C,
        horizon=cfg.horizon,
        out_dim=1,
        hidden_sizes=hidden_sizes,
        dropout=0.1,
        activation="relu",
    ).to(cfg.device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(cfg.device)
            yb = yb.to(cfg.device)

            pred = model(xb)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            train_loss += loss.item() * xb.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(cfg.device)
                yb = yb.to(cfg.device)
                pred = model(xb)
                val_loss += loss_fn(pred, yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch:03d} | train MSE {train_loss:.6f} | val MSE {val_loss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, scaler 





In [ ]:
@torch.no_grad()
def forecast_next(
    model: MLPForecaster,
    scaler: StandardScaler,
    recent_history: np.ndarray,     # shape (lookback,) or (lookback, C)
    target_channel: int = 0,
    device: Optional[str] = None,
) -> np.ndarray:
    """
    Returns forecast for the next horizon steps in original scale: shape (horizon,)
    Direct Multi-Step Forecasting
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    recent_history = np.asarray(recent_history, dtype=np.float32)
    if recent_history.ndim == 1:
        recent_history = recent_history[:, None]

    # scale and run
    x = scaler.transform(recent_history)  # (L, C)
    x = torch.from_numpy(x).unsqueeze(0).to(device)  # (1, L, C)

    model = model.to(device).eval()
    pred_scaled = model(x).cpu().numpy()[0, :, 0]  # (H,)

    # inverse-transform safely for any scaler type
    dummy = np.zeros((len(pred_scaled), recent_history.shape[1] if recent_history.ndim > 1 else 1))
    dummy[:, target_channel] = pred_scaled
    pred = scaler.inverse_transform(dummy)[:, target_channel]
    return pred

@torch.no_grad()
def recursive_forecast(
    model: MLPForecaster,
    scaler: StandardScaler,
    recent_history: np.ndarray,  # shape (lookback, C)
    horizon: int,
    target_channel: int = 0,
    device: Optional[str] = None,
) -> np.ndarray:
    """
    Recursively forecasts `horizon` steps using a 1-step-ahead model.
    Returns forecast in original scale: shape (horizon,)
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Prepare initial input
    recent_history = np.asarray(recent_history, dtype=np.float32)
    if recent_history.ndim == 1:
        recent_history = recent_history[:, None]
        
    current_x = recent_history.copy()
    
    # Scale input
    # Assuming univariate scaling as per training function
    current_x_scaled = scaler.transform(current_x) # (L, 1) or (L, C)
    
    preds_scaled = []
    
    model = model.to(device).eval()
    
    # Prepare tensor: (1, L, C)
    # We maintain the window as a numpy array and convert to tensor each step
    # or maintain as tensor. Numpy is easier for shifting.
    
    input_window = current_x_scaled.copy()
    
    for _ in range(horizon):
        x_tensor = torch.from_numpy(input_window).float().unsqueeze(0).to(device) # (1, L, C)
        
        # Predict 1 step
        y_pred = model(x_tensor) # (1, 1, 1) 
        val_pred = y_pred.item() 
        preds_scaled.append(val_pred)
        
        # Update input window: shift and append prediction
        input_window = np.roll(input_window, -1, axis=0)
        input_window[-1, target_channel] = val_pred
        
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    
    # Inverse transform
    # We need to create a dummy array matching the scaler's input shape
    dummy = np.zeros((len(preds_scaled), recent_history.shape[1]))
    dummy[:, target_channel] = preds_scaled[:, 0]
    
    preds = scaler.inverse_transform(dummy)[:, target_channel]
    return preds


In [ ]:
seeds = [42, 123, 999]

for seed in seeds:
    print(f"\n{'='*20} SEED {seed} {'='*20}")
    
    # --- 1. Direct Multi-Step ---
    print(">> Training Direct Multi-Step Model...")
    cfg_direct = TrainConfig(
        lookback=48,
        horizon=153, # Train to predict H steps at once
        batch_size=32,
        train_size=453,
        val_size=153,
        lr=1e-3,
        epochs=30,
        weight_decay=1e-4,
    )
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_direct, scaler_direct = train_mlp_forecaster(df[TARGET_COL].values, cfg_direct, target_channel=0)
    
    # Forecast Direct
    # Use history up to the test set start
    test_start_idx = -(cfg_direct.horizon) 
    history_start_idx = test_start_idx - cfg_direct.lookback
    recent_history_direct = df[TARGET_COL].values[history_start_idx : test_start_idx]
    
    pred_direct = forecast_next(model_direct, scaler_direct, recent_history_direct, target_channel=0) # horizon=153 implicitly from model

    # --- 2. Recursive Step-by-Step ---
    print("\n>> Training Recursive Model (1-step ahead)...")
    cfg_recursive = TrainConfig(
        lookback=48,
        horizon=1,   # Train to predict 1 step
        batch_size=32,
        train_size=453,
        val_size=153, # We still use same validation split logic, but model only predicts 1 step
        lr=1e-3,
        epochs=30,
        weight_decay=1e-4,
    )
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    model_recursive, scaler_recursive = train_mlp_forecaster(df[TARGET_COL].values, cfg_recursive, target_channel=0)

    # Forecast Recursive
    # Use history up to the test set start
    # Note: Recursive forecasting uses the SAME history window start, but iterates 153 times.
    # The `recent_history` is the same.
    
    # We want to forecast 153 steps into the future
    forecast_horizon = 153 
    
    pred_recursive = recursive_forecast(
        model_recursive, 
        scaler_recursive, 
        recent_history_direct, # Same history input
        horizon=forecast_horizon,
        target_channel=0
    )


    # --- Evaluation & Plotting ---
    test_values = df[TARGET_COL].values[-forecast_horizon:]
    
    # Direct Metrics
    rmse_direct = np.sqrt(mean_squared_error(test_values, pred_direct))
    mae_direct = mean_absolute_error(test_values, pred_direct)
    
    # Recursive Metrics
    rmse_recursive = np.sqrt(mean_squared_error(test_values, pred_recursive))
    mae_recursive = mean_absolute_error(test_values, pred_recursive)
    
    print(f"\nResults for Seed {seed}:")
    print(f"Direct:    RMSE={rmse_direct:.4f}, MAE={mae_direct:.4f}")
    print(f"Recursive: RMSE={rmse_recursive:.4f}, MAE={mae_recursive:.4f}")
    
    plt.figure(figsize=(12, 6))
    plt.plot(test_values, label='Actual', color='black', alpha=0.7)
    plt.plot(pred_direct, label=f'Direct (RMSE={rmse_direct:.2f})', color='blue')
    plt.plot(pred_recursive, label=f'Recursive (RMSE={rmse_recursive:.2f})', color='red', linestyle='--')
    plt.title(f'Direct vs Recursive Forecasting (Seed {seed})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
